# V2: With education-job features

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import random

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    classification_report
)

In [2]:
companies = pd.read_csv("companies.csv")
portfolio_edges = pd.read_csv("portfolio_edges.csv")
feature_matrix = pd.read_csv("feature_matrix.csv")
edu_job_features = pd.read_csv("edu_job_features.csv")

print("companies shape:", companies.shape)
print("portfolio_edges shape:", portfolio_edges.shape)
print("feature_matrix shape:", feature_matrix.shape)
print("edu_job_features shape:", edu_job_features.shape)

companies shape: (6704, 22)
portfolio_edges shape: (461543, 6)
feature_matrix shape: (6704, 37)
edu_job_features shape: (6704, 15)


In [3]:
companies = companies[companies["is_success"].notna()].copy()
companies["is_success"] = companies["is_success"].astype(int)

feature_matrix = feature_matrix[feature_matrix["is_success"].notna()].copy()
feature_matrix["is_success"] = feature_matrix["is_success"].astype(int)

print("Labeled companies:", companies.shape)
print("Labeled feature_matrix:", feature_matrix.shape)
print("Positive rate:", feature_matrix["is_success"].mean())

Labeled companies: (6593, 22)
Labeled feature_matrix: (6593, 37)
Positive rate: 0.16927043834369787


In [4]:
G = nx.Graph()

for _, row in portfolio_edges.iterrows():
    investor_id = "inv_" + str(row["vc_uuid"])
    company_id = str(row["portfolio_company_uuid"])
    G.add_edge(investor_id, company_id)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

Graph nodes: 160243
Graph edges: 353034


In [5]:
def random_walk(graph, start_node, walk_length=15):
    walk = [start_node]
    
    for _ in range(walk_length - 1):
        current_node = walk[-1]
        neighbors = list(graph.neighbors(current_node))
        
        if len(neighbors) == 0:
            break
        
        next_node = random.choice(neighbors)
        walk.append(next_node)
    
    return [str(node) for node in walk]

In [6]:
random.seed(42)
np.random.seed(42)

walks = []
all_nodes = list(G.nodes())

num_walks = 5
walk_length = 15

for _ in range(num_walks):
    random.shuffle(all_nodes)
    for node in all_nodes:
        walks.append(random_walk(G, node, walk_length=walk_length))

print("Number of walks:", len(walks))
print("Example walk:", walks[0][:10])

Number of walks: 801215
Example walk: ['0c6c101e-53f6-49c9-a57c-7f98a08301e4', 'inv_796e4250-e66a-1c7b-6ccf-5a4fbd3b1936', 'f08a3bfa-32a0-0015-645c-d893f2a2c2ae', 'inv_9183d3fb-c801-bc11-1594-04850e47cf60', 'ca9a4738-54ec-435a-95c9-1faabdf24b7d', 'inv_73633ee4-ea65-2967-6c5d-9b5fec7d2d5e', '9dc9cd36-acc4-43e0-96e9-b57eff2dbbed', 'inv_73633ee4-ea65-2967-6c5d-9b5fec7d2d5e', '73c10c11-d3e8-4c87-8378-14d959b3e7e1', 'inv_73633ee4-ea65-2967-6c5d-9b5fec7d2d5e']


In [7]:
embedding_model = Word2Vec(
    sentences=walks,
    vector_size=64,
    window=8,
    min_count=0,
    sg=1,
    workers=1,
    epochs=3,
    negative=10,
    seed=42
)

print("Embedding dimension:", embedding_model.vector_size)
print("Vocabulary size:", len(embedding_model.wv))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Embedding dimension: 64
Vocabulary size: 160243


In [8]:
embedding_dict = {}

company_ids = feature_matrix["company_uuid"].astype(str).unique()

for company_id in company_ids:
    if company_id in embedding_model.wv:
        embedding_dict[company_id] = embedding_model.wv[company_id]
    else:
        embedding_dict[company_id] = np.zeros(embedding_model.vector_size)

emb_df = pd.DataFrame.from_dict(embedding_dict, orient="index")
emb_df.index.name = "company_uuid"
emb_df.reset_index(inplace=True)

emb_df.columns = ["company_uuid"] + [f"emb_{i}" for i in range(emb_df.shape[1] - 1)]

print("emb_df shape:", emb_df.shape)
emb_df.head()

emb_df shape: (6593, 65)


,company_uuid,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63
0,fdd4bfe1-5f7b-e403-d100-91e9ec01f903,0.185357,-0.447779,0.194464,0.286596,-0.507258,-0.039527,-0.565614,-0.287946,-0.654519,...,0.072998,0.048468,0.636657,-0.260829,0.034060,-0.153023,-0.508651,0.322503,0.547409,1.081120
1,1ebaa59b-7035-75d2-316d-e7b35b8a82ed,0.557831,0.191456,0.264176,0.360012,-0.714454,0.073710,-0.087315,-0.731025,-1.133906,...,-0.829653,0.045984,0.801829,-0.559577,0.129349,-0.328260,0.728941,0.080259,0.094586,0.523972
2,4361b5e3-697b-325f-63e7-1814d63b865f,0.050295,-0.071381,0.673141,0.435259,-0.552265,-0.113035,0.273525,-0.360569,-0.536531,...,-0.088122,-0.058809,0.203975,-0.287169,-0.664536,-0.349433,0.262334,0.368523,-0.300297,0.901372
3,5e298e78-0999-fe40-4b90-c21ed1c90786,0.070279,-0.668616,0.483008,0.175601,-0.179456,0.606830,-0.768199,-0.542048,-0.272572,...,-0.358398,-0.714943,0.135357,-0.208963,0.803583,-0.631152,-0.313936,0.387316,0.876377,0.864584
4,16c38a41-e42c-e440-f372-d5f382be7662,0.726383,-1.109966,0.381321,0.017788,-0.091452,-0.850777,-0.471258,0.198428,0.198467,...,-0.040170,0.307964,1.112996,-0.948301,-0.900558,-0.454847,-0.634382,-0.062569,-0.299897,0.120405


In [9]:
leakage_cols = [
    "funding_total_usd",
    "log_funding",
    "num_funding_rounds",
    "company_age_months"
]

drop_cols = ["is_success"] + [col for col in leakage_cols if col in feature_matrix.columns]

feature_only_df = feature_matrix.drop(columns=drop_cols).copy()

# bool 转 int
for col in feature_only_df.columns:
    if feature_only_df[col].dtype == bool:
        feature_only_df[col] = feature_only_df[col].astype(int)

print("feature_only_df shape:", feature_only_df.shape)
print(feature_only_df.dtypes.head())

feature_only_df shape: (6593, 32)
company_uuid          object
employees_ordinal    float64
num_founders           int64
founder_has_phd        int64
founder_has_mba        int64
dtype: object


In [10]:
edu_job_df = edu_job_features.copy()

for col in edu_job_df.columns:
    if edu_job_df[col].dtype == bool:
        edu_job_df[col] = edu_job_df[col].astype(int)

edu_job_df = edu_job_df.fillna(0)

print("edu_job_df shape:", edu_job_df.shape)
print(edu_job_df.dtypes.head())

edu_job_df shape: (6704, 15)
company_uuid                  object
edu_data_available             int64
founder_top_univ_count         int64
founder_univ_degree_avg      float64
founder_univ_pagerank_max    float64
dtype: object


In [11]:
model_df = feature_matrix[["company_uuid", "is_success"]].copy()
model_df = model_df.merge(feature_only_df, on="company_uuid", how="left")
model_df = model_df.merge(emb_df, on="company_uuid", how="left")
model_df = model_df.merge(edu_job_df, on="company_uuid", how="left")
model_df = model_df.fillna(0)
print("model_df shape:", model_df.shape)
model_df.head()

model_df shape: (6593, 111)


,company_uuid,is_success,employees_ordinal,num_founders,founder_has_phd,founder_has_mba,max_founder_prior_jobs,avg_founder_prior_jobs,team_size,c_suite_count,...,co_alumni_investor_overlap,founder_alumni_network_size,founder_ex_faang_count,founder_ex_startup_count,founder_prior_org_pagerank_max,coworker_investor_overlap,founder_coworker_network_size,founder_industry_diversity,founder_investor_social_proximity,team_network_reach
0,fdd4bfe1-5f7b-e403-d100-91e9ec01f903,0,2.0,4,0,0,6,3.750000,0,0,...,0.0,570.0,0,0,0.000269,0.0,262.0,11,0.0,831.0
1,1ebaa59b-7035-75d2-316d-e7b35b8a82ed,1,7.0,3,1,0,11,6.000000,21,6,...,4.0,762.0,0,0,0.000047,0.0,35.0,10,1.0,797.0
2,4361b5e3-697b-325f-63e7-1814d63b865f,1,5.0,2,0,0,5,4.000000,6,3,...,0.0,53.0,0,0,0.001079,2.0,500.0,4,1.0,553.0
3,5e298e78-0999-fe40-4b90-c21ed1c90786,0,2.0,3,0,0,2,1.666667,3,1,...,0.0,0.0,0,0,0.000007,0.0,0.0,1,0.0,0.0
4,16c38a41-e42c-e440-f372-d5f382be7662,0,2.0,1,0,1,1,1.000000,0,0,...,0.0,3.0,0,0,0.000000,0.0,0.0,0,1.0,3.0


In [12]:
X = model_df.drop(columns=["company_uuid", "is_success"]).copy()
y = model_df["is_success"].astype(int).copy()

X.columns = X.columns.astype(str)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive rate:", y.mean())

X shape: (6593, 109)
y shape: (6593,)
Positive rate: 0.16927043834369787


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5274, 109)
Test shape: (1319, 109)


In [14]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
clf = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)

clf.fit(X_train_scaled, y_train)

y_pred_prob = clf.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_pred_prob >= 0.5).astype(int)

In [16]:
roc_auc = roc_auc_score(y_test, y_pred_prob)
pr_auc = average_precision_score(y_test, y_pred_prob)
accuracy = accuracy_score(y_test, y_pred)
baseline = y_test.mean()

print(f"ROC-AUC: {roc_auc:.4f} (test)")
print(f"PR-AUC: {pr_auc:.4f} (test) | Baseline: {baseline:.3f}")
print(f"Accuracy: {accuracy*100:.1f}%")

ROC-AUC: 0.8152 (test)
PR-AUC: 0.5596 (test) | Baseline: 0.169
Accuracy: 75.1%


In [17]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Success (0)", "Success (1)"]
))

                 precision    recall  f1-score   support

Not Success (0)       0.92      0.76      0.84      1096
    Success (1)       0.37      0.70      0.49       223

       accuracy                           0.75      1319
      macro avg       0.65      0.73      0.66      1319
   weighted avg       0.83      0.75      0.78      1319



In [18]:
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=["Not Success (0)", "Success (1)"],
    output_dict=True
)

report_df = pd.DataFrame(report_dict).T
report_df = report_df[["precision", "recall", "f1-score", "support"]]
report_df["support"] = report_df["support"].astype(int, errors="ignore")
report_df.round(3)

,precision,recall,f1-score,support
Not Success (0),0.925,0.763,0.836,1096
Success (1),0.373,0.695,0.486,223
accuracy,0.751,0.751,0.751,0
macro avg,0.649,0.729,0.661,1319
weighted avg,0.832,0.751,0.777,1319


In [19]:
np.save("y_test_v2.npy", np.array(y_test))
np.save("y_pred_v2.npy", np.array(y_pred_prob))